# 🌱 Crop Recommendation System — Exploratory Data Analysis

This notebook walks through the complete EDA and model training pipeline for the Crop Recommendation System.

**Dataset:** `../data/Crop_recommendation.csv`  
**Source:** https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

%matplotlib inline
sns.set_theme(style='whitegrid', palette='Greens_d')
print('Libraries loaded.')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/Crop_recommendation.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Columns:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())

## 2. Statistical Summary

In [ ]:
df.describe().round(2)

## 3. Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
counts = df['label'].value_counts()
ax.bar(counts.index, counts.values, color=sns.color_palette('Greens_d', len(counts)))
ax.set_title('Crop Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Crop')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(counts)

## 4. Correlation Heatmap

In [ ]:
feature_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[feature_cols].corr(), annot=True, fmt='.2f',
            cmap='YlGn', ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=30, color='#4CAF50', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
axes[-1].axis('off')
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Preprocessing

In [ ]:
df_clean = df.drop_duplicates()
print(f'Rows after dedup: {len(df_clean)}')

X = df_clean[feature_cols].values
y = df_clean['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {len(X_train)}')
print(f'Test size:  {len(X_test)}')

## 7. Train & Evaluate All Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
}

results = {}
for name, clf in models.items():
    clf.fit(X_train_sc, y_train)
    y_pred = clf.predict(X_test_sc)
    results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1':        f1_score(y_test, y_pred, average='weighted', zero_division=0),
    }

res_df = pd.DataFrame(results).T * 100
res_df = res_df.round(2)
res_df.columns = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-score (%)']
res_df

In [ ]:
ax = res_df.plot(kind='bar', figsize=(10, 5), ylim=(90, 105),
                 color=['#4CAF50','#2196F3','#FF9800','#9C27B0'],
                 edgecolor='white', alpha=0.85)
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Score (%)')
plt.xticks(rotation=15, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Best Model — Detailed Report

In [ ]:
best_name = max(results, key=lambda n: results[n]['f1'])
print(f'Best model: {best_name}  (F1 = {results[best_name]["f1"]*100:.2f}%)')

best_clf = models[best_name]
y_pred_best = best_clf.predict(X_test_sc)
print()
print(classification_report(y_test, y_pred_best, zero_division=0))

## 9. Confusion Matrix

In [ ]:
classes = sorted(np.unique(y_test))
cm = confusion_matrix(y_test, y_pred_best, labels=classes)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 10. Feature Importance (Random Forest)

In [ ]:
rf = models['Random Forest']
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([feature_cols[i] for i in idx],
       [importances[i] for i in idx],
       color=sns.color_palette('Greens_d', len(feature_cols)))
ax.set_title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_ylabel('Importance')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 11. Sample Prediction

Demonstrate a single prediction using the trained model.

In [ ]:
sample = np.array([[90, 42, 43, 25.5, 80.0, 6.5, 200.0]])
sample_sc = scaler.transform(sample)

crop = best_clf.predict(sample_sc)[0]
proba = best_clf.predict_proba(sample_sc)[0]
confidence = max(proba)

print(f'Input: N=90, P=42, K=43, Temp=25.5, Humidity=80, pH=6.5, Rainfall=200')
print(f'Recommended Crop : {crop.upper()}')
print(f'Confidence       : {confidence*100:.1f}%')

top5 = sorted(zip(best_clf.classes_, proba), key=lambda x: -x[1])[:5]
print('\nTop 5 probabilities:')
for c, p in top5:
    bar = '█' * int(p * 40)
    print(f'  {c:<15} {p*100:5.1f}%  {bar}')